In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# ==============================================================================
# 1. LOAD TRAINING AND TEST DATA
# ==============================================================================
print("1/5 Loading data...")
train_raw = pd.read_csv('data/vehicle_train_10000.csv')
test_raw = pd.read_csv('data/vehicle_test.csv')

# ==============================================================================
# 2. DOMAIN-SPECIFIC FEATURE ENGINEERING
# ==============================================================================
print("2/5 Creating engineered features...")
def create_super_features(df):
    d = df.copy()
    
    # Powertrain & Weight Dynamics
    d['power_to_weight'] = d['horsepower_hp'] / d['vehicle_weight_kg']
    d['torque_to_weight'] = d['torque_nm'] / d['vehicle_weight_kg']
    d['battery_to_weight'] = d['battery_capacity_kwh'] / d['vehicle_weight_kg']
    d['battery_to_accel'] = d['battery_capacity_kwh'] / (d['acceleration_0_100_kmh_s'] + 1e-5)
    d['power_per_cyl'] = d['horsepower_hp'] / d['engine_cylinders']
    d['torque_per_disp'] = d['torque_nm'] / (d['engine_displacement_l'] + 1e-5)
    
    # Exterior Dimensions & Geometry
    d['volume_est'] = (d['vehicle_length_mm'] * d['vehicle_width_mm'] * d['vehicle_height_mm']) / 1e9  # m^3
    d['density_est'] = d['vehicle_weight_kg'] / d['volume_est']
    d['wheelbase_to_length'] = d['wheelbase_mm'] / d['vehicle_length_mm']
    d['length_to_width'] = d['vehicle_length_mm'] / d['vehicle_width_mm']
    
    # Aerodynamics & Drag
    d['frontal_area'] = (d['vehicle_width_mm'] * d['vehicle_height_mm']) / 1e6
    d['drag_area'] = d['drag_coefficient'] * d['frontal_area']
    
    # Energy, Braking & Clearance
    d['kinetic_energy'] = 0.5 * d['vehicle_weight_kg'] * ((100 / 3.6) ** 2)
    d['braking_force'] = d['kinetic_energy'] / (d['braking_distance_m'] + 1e-5)
    d['avg_fuel_efficiency'] = (d['city_fuel_efficiency_kmpl'] + d['highway_fuel_efficiency_kmpl']) / 2.0
    d['fuel_tank_to_weight'] = d['fuel_tank_capacity_l'] / d['vehicle_weight_kg']
    d['low_slung_ratio'] = d['ground_clearance_mm'] / (d['vehicle_height_mm'] + 1e-5)
    
    return d

train_fe = create_super_features(train_raw)
test_fe = create_super_features(test_raw)

# Separate features and target
feat_cols = [c for c in train_fe.columns if c not in ['vehicle_id', 'vehicle_class']]
X_train = train_fe[feat_cols]
y_train = train_fe['vehicle_class']
X_test = test_fe[feat_cols]

print(f"Features created: {len(feat_cols)}")
print(f"Training shape: {X_train.shape}, Test shape: {X_test.shape}")

# ==============================================================================
# 3. INITIALIZE MODELS (TREE ENSEMBLE + LINEAR HYPERPLANE)
# ==============================================================================
print("3/5 Training individual models on full dataset...")

# Model 1: Gini Bagging Classifier (300 trees, feature subsampling)
print("  - Fitting Gini Bagging (300 trees)...")
m_bag_gini = BaggingClassifier(
    estimator=DecisionTreeClassifier(max_depth=16, min_samples_leaf=2, random_state=42),
    n_estimators=300,
    max_features=0.85,
    random_state=42,
    n_jobs=-1
)
m_bag_gini.fit(X_train, y_train)
p_bag_gini = m_bag_gini.predict_proba(X_test)

# Model 2: Entropy Bagging Classifier (300 trees, feature subsampling)
print("  - Fitting Entropy Bagging (300 trees)...")
m_bag_ent = BaggingClassifier(
    estimator=DecisionTreeClassifier(criterion='entropy', max_depth=16, min_samples_leaf=2, random_state=101),
    n_estimators=300,
    max_features=0.85,
    random_state=101,
    n_jobs=-1
)
m_bag_ent.fit(X_train, y_train)
p_bag_ent = m_bag_ent.predict_proba(X_test)

# Model 3: Random Forest (500 trees)
print("  - Fitting Random Forest (500 trees)...")
m_rf = RandomForestClassifier(
    n_estimators=500,
    min_samples_split=4,
    random_state=42,
    n_jobs=-1
)
m_rf.fit(X_train, y_train)
p_rf = m_rf.predict_proba(X_test)

# Model 4: ExtraTrees (500 trees)
print("  - Fitting ExtraTrees (500 trees)...")
m_et = ExtraTreesClassifier(
    n_estimators=500,
    min_samples_split=4,
    random_state=42,
    n_jobs=-1
)
m_et.fit(X_train, y_train)
p_et = m_et.predict_proba(X_test)

# Model 5: Scaled Regularized Logistic Regression (Smooth Hyperplane)
print("  - Fitting Scaled Logistic Regression...")
m_lr = make_pipeline(
    StandardScaler(),
    LogisticRegression(C=1.0, max_iter=2000, random_state=42)
)
m_lr.fit(X_train, y_train)
p_lr = m_lr.predict_proba(X_test)

# ==============================================================================
# 4. HYBRID PROBABILITY BLENDING
# ==============================================================================
print("4/5 Blending soft probabilities (40% Tree Ensemble + 60% Linear Boundary)...")
# Average the 4 tree-based bagging models (1,600 trees total)
p_tree_ensemble = (p_bag_gini * 0.25) + (p_bag_ent * 0.25) + (p_rf * 0.25) + (p_et * 0.25)

# Blend tree ensemble with linear model probabilities
p_hybrid = (p_tree_ensemble * 0.40) + (p_lr * 0.60)
preds_hybrid = np.argmax(p_hybrid, axis=1)

# ==============================================================================
# 5. GENERATE & VERIFY SUBMISSION FILE
# ==============================================================================
print("5/5 Exporting submission file...")
output_filename = 'submission_hybrid_super_blend.csv'
sub = pd.DataFrame({
    'vehicle_id': test_raw['vehicle_id'],
    'vehicle_class': preds_hybrid
})
sub.to_csv(output_filename, index=False)

print(f"\n[SUCCESS] File '{output_filename}' generated successfully!")
print("Submission Preview (First 5 rows):")
print(sub.head())

print("\nPredicted Class Balance (%):")
print(sub['vehicle_class'].value_counts(normalize=True).round(4) * 100)



1/5 Loading data...
2/5 Creating engineered features...
Features created: 37
Training shape: (10000, 37), Test shape: (2500, 37)
3/5 Training individual models on full dataset...
  - Fitting Gini Bagging (300 trees)...
  - Fitting Entropy Bagging (300 trees)...
  - Fitting Random Forest (500 trees)...
  - Fitting ExtraTrees (500 trees)...
  - Fitting Scaled Logistic Regression...
4/5 Blending soft probabilities (40% Tree Ensemble + 60% Linear Boundary)...
5/5 Exporting submission file...

[SUCCESS] File 'submission_hybrid_super_blend.csv' generated successfully!
Submission Preview (First 5 rows):
   vehicle_id  vehicle_class
0       10001              0
1       10002              0
2       10003              1
3       10004              2
4       10005              1

Predicted Class Balance (%):
vehicle_class
1    33.36
0    33.32
2    33.32
Name: proportion, dtype: float64
